In [1]:
import pandas as pd
import numpy as np


cols=['ride_id','rideable_type','started_at','ended_at','start_lat','start_lng','end_lat','end_lng','member_casual' ]
df=pd.read_csv('dataset_maestro_ciclistas.csv', usecols=cols)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5552994 entries, 0 to 5552993
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   ride_id        object 
 1   rideable_type  object 
 2   started_at     object 
 3   ended_at       object 
 4   start_lat      float64
 5   start_lng      float64
 6   end_lat        float64
 7   end_lng        float64
 8   member_casual  object 
dtypes: float64(4), object(5)
memory usage: 381.3+ MB


In [2]:
total_reg = len(df)
total_reg

5552994

El total de registros en todo el dataset es 5552994

In [3]:
#Convertir las columnas de fecha a formato datetime
df['started_at']=pd.to_datetime(df['started_at'], errors='coerce')
df['ended_at']=pd.to_datetime(df['ended_at'], errors='coerce')

In [4]:
print(df[['started_at','ended_at']].head())
print(df[['started_at','ended_at']].dtypes)
print(df['started_at'].isna().sum())
print(df['ended_at'].isna().sum())
print(df[['started_at','ended_at']].shape)

               started_at                ended_at
0 2025-01-21 17:23:54.538 2025-01-21 17:37:52.015
1 2025-01-11 15:44:06.795 2025-01-11 15:49:11.139
2 2025-01-02 15:16:27.730 2025-01-02 15:28:03.230
3 2025-01-23 08:49:05.814 2025-01-23 08:52:40.047
4 2025-01-16 08:38:32.338 2025-01-16 08:41:06.767
started_at    datetime64[ns]
ended_at      datetime64[ns]
dtype: object
0
0
(5552994, 2)


In [5]:
#Se crea la columna de duración en minutos
df['duration_min']=(df['ended_at'] - df['started_at']).dt.total_seconds() / 60

In [6]:
df['duration_min'].describe(percentiles=[.25, .5, .75, .90, .95, .99])

count    5.552994e+06
mean     1.602772e+01
std      5.511650e+01
min     -5.479480e+01
25%      5.394933e+00
50%      9.426058e+00
75%      1.656325e+01
90%      2.821158e+01
95%      3.970543e+01
99%      8.903635e+01
max      1.574900e+03
Name: duration_min, dtype: float64

In [7]:
df['duration_min'].isna().sum()

0

In [8]:
#Se valida el % de duraciones negativas

# Crear máscara lógica (no crea copia pesada)
tiempo_valido = df["duration_min"] > 0  

# Contar válidos e inválidos
registros_validos = tiempo_valido.sum()
registros_eliminados = total_reg - registros_validos

porcentaje_eliminado = (registros_eliminados / total_reg) * 100

print("Total inicial:", total_reg)
print("Registros eliminados:", registros_eliminados)
print("Porcentaje eliminado:", round(porcentaje_eliminado,2), "%")




Total inicial: 5552994
Registros eliminados: 29
Porcentaje eliminado: 0.0 %


Se comprueba al respecto de duraciones invalidas se eliminan 29 registros:
Total inicial: 5552994
Registros eliminados: 29
Porcentaje eliminado: 0.0 %

In [9]:
#Se eliminan los registros con duración negativa
df = df.loc[df["duration_min"] > 0].copy()

In [10]:
df['duration_min'].describe()

count    5.552965e+06
mean     1.602802e+01
std      5.511648e+01
min      7.666667e-04
25%      5.395000e+00
50%      9.426100e+00
75%      1.656328e+01
max      1.574900e+03
Name: duration_min, dtype: float64

In [11]:
#Se crean columnas, drivers del analisis exploratorio
df['day_of_week']=df['started_at'].dt.day_name()
df['month']=df['started_at'].dt.month_name()
df['hour']=df['started_at'].dt.hour
df['is_weekend']=df['day_of_week'].isin(['Saturday', 'Sunday'])

In [12]:
#Se crea columna
bins=[0, 3, 10, 20, 35, float('inf')]

labels=['Muy Corto (0-3 min)',
        'Corto (3-10 min)',
          'Medio (10-20 min)', 
          'Largo (20-35 min)',
            'Muy Largo (35+ min)']

df['ride_length']=pd.cut(df['duration_min'], bins=bins, labels=labels, right=True, include_lowest=True)

In [13]:
#Se crea columna
bins_hora=[0, 6, 12, 18, 24]

labels_hora=['Madrugada (0-6)', 
             'Mañana (6-12)', 
             'Tarde (12-18)', 
             'Noche (18-24)']

df["time_of_day"]=pd.cut(df['hour'], bins=bins_hora, labels=labels_hora, right=False, include_lowest=True)

In [14]:
#Se crea columna de estación del año
season={
    'December': 'Invierno',
    'January': 'Invierno',
    'February': 'Invierno',
    'March': 'Primavera',
    'April': 'Primavera',
    'May': 'Primavera',
    'June': 'Verano',
    'July': 'Verano',
    'August': 'Verano',
    'September': 'Otoño',
    'October': 'Otoño',
    'November': 'Otoño'
}

df['season']=df['month'].map(season)


In [15]:
#Se crean columnas de latitud y longitud parte 1 Haversine distance_km
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371  # radio Tierra en km-->estandar
    
    lat1_rad = np.radians(lat1) #Convertir grados a radianes
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad)*np.cos(lat2_rad)*np.sin(dlon/2)**2 #-->formula de Haversive
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c # distancia = radio × ángulo

# crear distancia solo donde hay coordenadas completas
mask_valid = df['end_lat'].notna()

df.loc[mask_valid, 'distance_km'] = haversine_vectorized(
    df.loc[mask_valid, 'start_lat'],
    df.loc[mask_valid, 'start_lng'],
    df.loc[mask_valid, 'end_lat'],
    df.loc[mask_valid, 'end_lng']
)

In [16]:
#Se crean columnas de latitud y longitud parte 2 Categorizar distancia_km
bins_dist=[0,1,3,6,15,np.inf]

labels_dist=['Muy corto: 0-1km',
             'Corto: 1-3km',
             'Medio: 3-6km',
             'Largo: 6-15km',
             'Muy largo: 15+km']

df['distance_km']= pd.cut(df['distance_km'],bins=bins_dist, labels=labels_dist, include_lowest=True)

In [17]:
#Crosstab de rideable_type por member_casual
tabla_mxr=pd.crosstab(df['member_casual'], 
            df['rideable_type'],
            normalize="index")*100

dif_ppmxr = tabla_mxr.loc["casual"] - tabla_mxr.loc["member"]
dif_ppmxr

rideable_type
classic_bike    -2.248341
electric_bike    2.248341
dtype: float64

In [18]:
#Crosstab member_casual y hour
tabla_mxh=pd.crosstab(df['member_casual'],
                      df['hour'],
                      normalize="index")*100
tabla_mxh
dif_ppmxh = tabla_mxh.loc["casual"] - tabla_mxh.loc["member"]
dif_ppmxh

hour
0     1.036990
1     0.687841
2     0.498023
3     0.239122
4     0.114891
5    -0.390729
6    -1.511517
7    -3.138991
8    -3.698382
9    -1.161268
10    0.236116
11    0.729866
12    1.076250
13    1.383375
14    1.646734
15    1.042056
16   -0.620462
17   -1.208927
18   -0.133642
19    0.328038
20    0.416220
21    0.641160
22    0.902132
23    0.885106
dtype: float64

In [19]:
#Crosstab member_casual y time_of_day
tabla_mxt=pd.crosstab(df['member_casual'],
                      df['time_of_day'],
                      normalize='index')*100

dif_ppmxt=tabla_mxt.loc['casual']-tabla_mxt.loc['member']
dif_ppmxt

time_of_day
Madrugada (0-6)    2.186138
Mañana (6-12)     -8.544177
Tarde (12-18)      3.319025
Noche (18-24)      3.039013
dtype: float64

In [20]:
#Crosstab member_casual y ride_length

tabla_mxrl=pd.crosstab(df['member_casual'],
                      df['ride_length'],
                      normalize='index')*100

dif_ppmxrl=tabla_mxrl.loc['casual']-tabla_mxrl.loc['member']
dif_ppmxrl

ride_length
Muy Corto (0-3 min)    -1.337596
Corto (3-10 min)      -12.391971
Medio (10-20 min)       0.664526
Largo (20-35 min)       4.536863
Muy Largo (35+ min)     8.528179
dtype: float64

In [21]:
#Crosstab member_casual y month
tabla_mxm=pd.crosstab(df['member_casual'],
                      df['month'],
                      normalize='index')*100
dif_ppmxm=tabla_mxm.loc['casual']-tabla_mxm.loc['member']
dif_ppmxm

month
April       -1.913567
August       4.165936
December    -1.758912
February    -2.105387
January     -2.016447
July         3.786518
June         3.719066
March       -1.679330
May          0.139938
November    -2.288269
October     -0.672556
September    0.623011
dtype: float64

In [22]:
#Crosstab member_casual y season
tabla_mxs=pd.crosstab(df['member_casual'],
                      df['season'],
                      normalize='index')*100

dif_ppmxs=tabla_mxs.loc['casual']-tabla_mxs.loc['member']
dif_ppmxs

season
Invierno     -5.880746
Otoño        -2.337814
Primavera    -3.452960
Verano       11.671520
dtype: float64

In [23]:
#Crosstab member_casual y day_of_week
tabla_mxw=pd.crosstab(df['member_casual'],
                      df['day_of_week'],
                      normalize='index')*100
dif_ppmxw=tabla_mxw.loc['casual']-tabla_mxw.loc['member']
dif_ppmxw

day_of_week
Friday       1.121430
Monday      -2.733021
Saturday     8.046350
Sunday       5.835118
Thursday    -3.304063
Tuesday     -4.563419
Wednesday   -4.402396
dtype: float64

In [24]:
#Crosstab member_casual y is_weekend
tabla_mxiw=pd.crosstab(df['member_casual'],
                       df['is_weekend'],
                       normalize='index')*100
dif_ppmxiw=tabla_mxiw.loc['casual']-tabla_mxiw.loc['member']
dif_ppmxiw

is_weekend
False   -13.881468
True     13.881468
dtype: float64

In [25]:
#Crosstab member_casual y distance_km
tabla_mxd=pd.crosstab(df['member_casual'],
                      df['distance_km'],
                      normalize='index')*100

dif_ppmxd=tabla_mxd.loc['casual']-tabla_mxd.loc['member']
dif_ppmxd

distance_km
Muy corto: 0-1km    0.541016
Corto: 1-3km        1.015999
Medio: 3-6km       -1.197767
Largo: 6-15km      -0.397968
Muy largo: 15+km    0.038720
dtype: float64

In [26]:
impactos = {
    "rideable_type": dif_ppmxr.abs().max(),
    "hour": dif_ppmxh.abs().max(),
    "time_of_day": dif_ppmxt.abs().max(),
    "ride_length": dif_ppmxrl.abs().max(),
    "day_of_week": dif_ppmxw.abs().max(),
    "is_weekend": dif_ppmxiw.abs().max(),
    "month": dif_ppmxm.abs().max(),
    "season": dif_ppmxs.abs().max(),
    "distance_km": dif_ppmxd.abs().max()
}

impactos

ranking = (
    pd.Series(impactos, name="impacto_pp")
    .sort_values(ascending=False)
    .to_frame()
)

ranking

,impacto_pp
is_weekend,13.881468
ride_length,12.391971
season,11.671520
time_of_day,8.544177
day_of_week,8.046350
month,4.165936
hour,3.698382
rideable_type,2.248341
distance_km,1.197767


In [28]:
#df.to_csv("dataset_limpio.csv", index=False)


In [29]:
df.to_csv(
    "dataset_limpiov2.csv",
    index=False,
    sep=";",
    encoding="utf-8",
    decimal=".",
)

In [30]:
df.to_pickle("dataset_limpio.pkl")

In [31]:
df_loaded = pd.read_pickle("dataset_limpio.pkl")
df_loaded["duration_min"].describe()

count    5.552965e+06
mean     1.602802e+01
std      5.511648e+01
min      7.666667e-04
25%      5.395000e+00
50%      9.426100e+00
75%      1.656328e+01
max      1.574900e+03
Name: duration_min, dtype: float64

In [32]:
df[df["duration_min"] > 10000]

,ride_id,rideable_type,started_at,ended_at,start_lat,start_lng,end_lat,end_lng,member_casual,duration_min,day_of_week,month,hour,is_weekend,ride_length,time_of_day,season,distance_km


In [ ]:
'''import csv

df.to_csv(
    "dataset_limpio_seguro.csv",
    index=False,
    sep="|",
    quoting=csv.QUOTE_ALL,
    encoding="utf-8",
    escapechar="\\"
)'''